In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import einops

In [2]:
import lightning_scripts.lightning_ssl_matched_speech_in_noise as lightning 
importlib.reload(lightning)

LitAudioSSL = lightning.LitAudioSSL

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/barlow_word_resnet18_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment_avg_pool.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['hparas']['global_batch_size'] = 64
config['model']['arch_kwargs']['frame_wise_loss'] = True
config['num_gpus'] = 1 

model = LitAudioSSL(config)


### Fold time into batch dimension 

Interleave time points for each representation $x \in \Bbb R ^{b \times c \times  f \times  t} \to z \in \Bbb R ^{(b \times t) \times c \times  f}$ such that $x_{i,c,f,t} = z_{j,c,f}$, where $j$ indexes over the interleaved batch and time dimension

In [3]:
# x = torch.randn(1,1,211,400)
x = torch.randn(1,1,40000)
final, inv_out, all_outputs = model(x)
# ssl_feature = all_outputs['layer4']

## Fold time into batch dimension - time is collated 
# time_flattened = einops.rearrange(out_rep,"b c f t -> (b t) c f")

### Dev lightning module

In [4]:
import logging
logging.getLogger('sox').setLevel(logging.ERROR)

trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)

trainer.fit(model)


/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name            | Type                       | Params | Mode 
-----------------------------------------------------------------------
0 | audio_rep       | AudioToAudioRepresentation | 0      | train
1 | model           | ModelWithFrontEnd          | 12.6 M 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Rank 0 N training batches 317
This is the first step after restoring from a checkpoint!
z_1.shape=torch.Size([1664, 512])
z_2.shape=torch.Size([1664, 512])
z_1.shape=torch.Size([1664, 512])
z_2.shape=torch.Size([1664, 512])


Training: |          | 0/? [00:00<?, ?it/s]

z_1.shape=torch.Size([1664, 512])
z_2.shape=torch.Size([1664, 512])
z_1.shape=torch.Size([1664, 512])
z_2.shape=torch.Size([1664, 512])
z_1.shape=torch.Size([1664, 512])
z_2.shape=torch.Size([1664, 512])
z_1.shape=torch.Size([1664, 512])
z_2.shape=torch.Size([1664, 512])



Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined